#Group Project
## MovieLens 32M with DuckDB and Parquet (Phase 1: local analytical layer)

Datasets:
- [MovieLens 32M](https://grouplens.org/datasets/movielens) — 32M ratings, 2M tags, ~87.6K movies, ~201K users.
- [IMDb non-commercial datasets](https://datasets.imdbws.com) — title basics (~12.9M titles: type, year, runtime), joined to MovieLens through `links.csv`. Only MovieLens ratings are analysed.

## A. Setup

In [110]:
from pathlib import Path
import time
import duckdb
from IPython.display import display

ROOT = Path.cwd().resolve()
RAW_DIR = ROOT / 'raw'
PARQUET_DIR = ROOT / 'parquet'
DB_PATH = ROOT / 'movielens.duckdb'

RAW_DIR.mkdir(exist_ok=True)
PARQUET_DIR.mkdir(exist_ok=True)

PARQUET_FILES = {
    'movies.csv':  'movies.parquet',
    'ratings.csv': 'ratings.parquet',
    'tags.csv':    'tags.parquet',
    'links.csv':   'links.parquet',
    'title.basics.tsv':  'imdb_basics.parquet',
}

def duck_path(path: Path) -> str:
    return path.resolve().as_posix().replace("'", "''")

print('DuckDB:', duckdb.__version__)
print('Workspace:', ROOT)

DuckDB: 1.5.6
Workspace: /Users/nuno/Documents/Pg/big data/project


## B. Check the raw data

In [111]:
missing = [n for n in PARQUET_FILES if not (RAW_DIR / n).is_file()]
if missing:
    raise FileNotFoundError(f'Missing files in {RAW_DIR}: {missing}')

for name in PARQUET_FILES:
    print(f'{name:15s} {(RAW_DIR / name).stat().st_size / 1_000_000:8.2f} MB')

movies.csv          4.24 MB
ratings.csv       877.08 MB
tags.csv           72.35 MB
links.csv           1.95 MB
title.basics.tsv  1116.85 MB


## C. Convert CSV to Parquet
The conversion creates files, not DuckDB tables. Files that already exist are skipped. The IMDb file (1.1 GB of TSV) is converted the same way.

In [112]:
con = duckdb.connect()

# IMDb files are tab-separated, have no quoting, and use \\N for NULL
TSV_OPTIONS = "delim='\\t', quote='', nullstr='\\N'"

def read_raw(name):
    path = duck_path(RAW_DIR / name)
    extra = f', {TSV_OPTIONS}' if name.endswith('.tsv') else ''
    return f"read_csv_auto('{path}'{extra})"

for csv_name, parquet_name in PARQUET_FILES.items():
    target = PARQUET_DIR / parquet_name
    if target.exists():
        continue
    con.execute(f"""
        COPY (SELECT * FROM {read_raw(csv_name)})
        TO '{duck_path(target)}' (FORMAT PARQUET, COMPRESSION ZSTD);
    """)

print(f'{"file":20s} {"CSV MB":>9s} {"Parquet MB":>11s} {"ratio":>7s}')
for csv_name, parquet_name in PARQUET_FILES.items():
    c = (RAW_DIR / csv_name).stat().st_size / 1_000_000
    p = (PARQUET_DIR / parquet_name).stat().st_size / 1_000_000
    print(f'{parquet_name:20s} {c:9.1f} {p:11.1f} {c/p:6.1f}x')

file                    CSV MB  Parquet MB   ratio
movies.parquet             4.2         1.3    3.3x
ratings.parquet          877.1       157.8    5.6x
tags.parquet              72.4        13.1    5.5x
links.parquet              2.0         0.7    2.6x
imdb_basics.parquet     1116.8       279.6    4.0x


## D. Register external relations
A persistent `VIEW` over `read_parquet()` keeps the metadata in `movielens.duckdb` and the data in the Parquet files.

In [113]:
con.close()
con = duckdb.connect(str(DB_PATH))

for csv_name, parquet_name in PARQUET_FILES.items():
    view_name = Path(parquet_name).stem
    con.execute(f"""
        CREATE OR REPLACE VIEW {view_name} AS
        SELECT * FROM read_parquet('{duck_path(PARQUET_DIR / parquet_name)}');
    """)

def q(sql):
    return con.sql(sql).df()

display(q("""
SELECT table_name, table_type
FROM information_schema.tables
WHERE table_schema = 'main'
ORDER BY table_name;
"""))

,table_name,table_type
0,imdb_basics,VIEW
1,links,VIEW
2,movie_stats,VIEW
3,movies,VIEW
4,movies_imdb,VIEW
5,ratings,VIEW
6,tags,VIEW


## E. Know the dataset

In [114]:
display(q("""
SELECT 'ratings' AS relation, COUNT(*) AS rows FROM ratings
UNION ALL SELECT 'movies', COUNT(*) FROM movies
UNION ALL SELECT 'tags',   COUNT(*) FROM tags
UNION ALL SELECT 'links',  COUNT(*) FROM links
UNION ALL SELECT 'imdb_basics',  COUNT(*) FROM imdb_basics
UNION ALL SELECT 'users (distinct)', COUNT(DISTINCT userId) FROM ratings;
"""))

display(q("""
SELECT MIN(rating) AS min_rating, MAX(rating) AS max_rating,
       to_timestamp(MIN(timestamp)) AS first_rating,
       to_timestamp(MAX(timestamp)) AS last_rating
FROM ratings;
"""))

,relation,rows
0,ratings,32000204
1,movies,87585
2,tags,2000072
3,links,87585
4,imdb_basics,12861340
5,users (distinct),200948


,min_rating,max_rating,first_rating,last_rating
0,0.5,5.0,1995-01-09 12:46:44+01:00,2023-10-13 03:29:07+01:00


## F. Analytical questions
1. How many movies are there per genre, and how are they rated?
2. How are ratings distributed?
3. Which movies have the highest average rating (with enough ratings to trust it)?
4. How did rating activity evolve over time?
5. Which tags are used the most?
6. Is activity concentrated in a few users?

### Q1 — Movies per genre
Genres are stored as `Adventure|Comedy|...`, so we split them first.

In [115]:
display(q("""
WITH g AS (
    SELECT movieId, UNNEST(STR_SPLIT(genres, '|')) AS genre
    FROM movies
    WHERE genres <> '(no genres listed)'
)
SELECT genre,
       COUNT(DISTINCT movieId) AS movies,
       COUNT(r.rating)         AS ratings,
       ROUND(AVG(r.rating), 2) AS avg_rating
FROM g
LEFT JOIN ratings r USING (movieId)
GROUP BY genre
ORDER BY movies DESC;
"""))

,genre,movies,ratings,avg_rating
0,Drama,34175,13973271,3.68
1,Comedy,23124,11206926,3.43
2,Thriller,11823,8679464,3.53
3,Romance,10369,5524615,3.55
4,Action,9668,9665213,3.48
5,Documentary,9363,427353,3.69
6,Horror,8654,2492315,3.31
7,Crime,6976,5373051,3.69
8,Adventure,5402,7590522,3.52
9,Sci-Fi,4907,5717337,3.49


### Q2 — Rating distribution

In [116]:
display(q("""
SELECT rating,
       COUNT(*) AS n,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM ratings
GROUP BY rating
ORDER BY rating;
"""))

,rating,n,pct
0,0.5,525132,1.64
1,1.0,946675,2.96
2,1.5,531063,1.66
3,2.0,2028622,6.34
4,2.5,1685386,5.27
5,3.0,6054990,18.92
6,3.5,4290105,13.41
7,4.0,8367654,26.15
8,4.5,2974000,9.29
9,5.0,4596577,14.36


### Q3 — Highest-rated movies
Only movies with at least 1,000 ratings.

In [117]:
display(q("""
SELECT m.title,
       COUNT(*) AS n_ratings,
       ROUND(AVG(r.rating), 3) AS avg_rating
FROM ratings r
JOIN movies m USING (movieId)
GROUP BY m.movieId, m.title
HAVING COUNT(*) >= 1000
ORDER BY avg_rating DESC, n_ratings DESC
LIMIT 15;
"""))

,title,n_ratings,avg_rating
0,Planet Earth II (2016),1956,4.447
1,Planet Earth (2006),2948,4.444
2,Band of Brothers (2001),2811,4.427
3,"Shawshank Redemption, The (1994)",102929,4.405
4,"Godfather, The (1972)",66440,4.317
5,Parasite (2019),11670,4.312
6,Blue Planet II (2017),1163,4.300
7,Twin Peaks (1989),1140,4.299
8,"Usual Suspects, The (1995)",67750,4.265
9,12 Angry Men (1957),21863,4.265


### Q4 — Rating activity per year

In [118]:
display(q("""
SELECT YEAR(to_timestamp(timestamp)) AS year,
       COUNT(*) AS ratings,
       COUNT(DISTINCT userId) AS active_users
FROM ratings
GROUP BY year
ORDER BY year;
"""))

,year,ratings,active_users
0,1995,4,2
1,1996,1571368,25305
2,1997,685388,11073
3,1998,301691,3662
4,1999,1174629,9227
5,2000,1912322,14668
6,2001,1160098,10224
7,2002,849762,7788
8,2003,1011000,8406
9,2004,1139068,8536


### Q5 — Most used tags

In [119]:
display(q("""
SELECT tag, COUNT(*) AS uses
FROM tags
GROUP BY tag
ORDER BY uses DESC
LIMIT 15;
"""))

,tag,uses
0,sci-fi,10996
1,atmospheric,9589
2,action,8473
3,comedy,8139
4,funny,7467
5,surreal,7231
6,visually appealing,7090
7,based on a book,6617
8,twist ending,6521
9,dark comedy,6053


### Q6 — Long tail of user activity
Users grouped by how many ratings they wrote.

In [120]:
display(q("""
WITH per_user AS (
    SELECT userId, COUNT(*) AS n FROM ratings GROUP BY userId
)
SELECT CASE
         WHEN n < 50   THEN '1. under 50'
         WHEN n < 200  THEN '2. 50-199'
         WHEN n < 1000 THEN '3. 200-999'
         WHEN n < 5000 THEN '4. 1000-4999'
         ELSE               '5. 5000+'
       END AS ratings_per_user,
       COUNT(*) AS users,
       SUM(n)   AS total_ratings,
       ROUND(100.0 * SUM(n) / SUM(SUM(n)) OVER (), 2) AS pct_of_ratings
FROM per_user
GROUP BY ratings_per_user
ORDER BY ratings_per_user;
"""))

,ratings_per_user,users,total_ratings,pct_of_ratings
0,1. under 50,72604,2296720.0,7.18
1,2. 50-199,86147,8711332.0,27.22
2,3. 200-999,38576,15154023.0,47.36
3,4. 1000-4999,3585,5586235.0,17.46
4,5. 5000+,36,251894.0,0.79


## G. IMDb integration
`links.csv` maps each MovieLens `movieId` to an IMDb id. IMDb identifiers are `tt` + 7 digits, so the join key is `'tt' || imdbId`.

We create two reusable views:
- `movie_stats` — average MovieLens rating and number of ratings per movie.
- `movies_imdb` — MovieLens movies with IMDb type, year and runtime.

In [121]:
con.execute("""
CREATE OR REPLACE VIEW movie_stats AS
SELECT movieId, COUNT(*) AS ml_ratings, AVG(rating) AS ml_avg_rating
FROM ratings
GROUP BY movieId;
""")

con.execute("""
CREATE OR REPLACE VIEW movies_imdb AS
SELECT m.movieId, m.title, m.genres,
       b.titleType, b.startYear, b.runtimeMinutes
FROM movies m
JOIN links l USING (movieId)
JOIN imdb_basics b ON b.tconst = 'tt' || l.imdbId;
""")

display(q("""
SELECT COUNT(*) AS movielens_movies,
       COUNT(e.movieId) AS matched_in_imdb,
       ROUND(100.0 * COUNT(e.movieId) / COUNT(*), 2) AS pct_matched
FROM movies m
LEFT JOIN movies_imdb e ON e.movieId = m.movieId;
"""))

,movielens_movies,matched_in_imdb,pct_matched
0,87585,87355,99.74


### Q7 — Which decades produced the best-rated movies?
Release year comes from IMDb (MovieLens only has it inside the title). Only feature films with at least 100 MovieLens ratings.

In [122]:
display(q("""
SELECT (e.startYear // 10) * 10 AS decade,
       COUNT(*) AS movies,
       ROUND(AVG(s.ml_avg_rating), 2) AS avg_ml_rating,
       ROUND(AVG(e.runtimeMinutes), 0) AS avg_runtime_min
FROM movies_imdb e
JOIN movie_stats s USING (movieId)
WHERE e.titleType = 'movie' AND s.ml_ratings >= 100 AND e.startYear IS NOT NULL
GROUP BY decade
ORDER BY decade;
"""))

,decade,movies,avg_ml_rating,avg_runtime_min
0,1910,4,3.35,145.0
1,1920,49,3.67,92.0
2,1930,162,3.65,97.0
3,1940,245,3.65,103.0
4,1950,352,3.60,106.0
5,1960,489,3.57,114.0
6,1970,634,3.44,110.0
7,1980,1299,3.17,105.0
8,1990,2314,3.19,106.0
9,2000,2963,3.23,106.0
